In [1]:
import numpy as np
import numba as nb
import scipy.stats as stats
import torch
import matplotlib.pyplot as plt
import pickle
import yaml

from sbi import utils as utils
from sbi import analysis as analysis
from sbi.utils.user_input_checks import process_simulator
from sbi.inference.posteriors import DirectPosterior

/home/dingeldein/Venvs/fret_sbi/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from functools import partial
import simulator as sim

In [ ]:
with open("conf/simulator/test.yaml", 'r') as f:
    config = yaml.safe_load(f)

sim_cfg = config['simulator']
prior_cfg = config['prior']

In [ ]:
def bin_along_time(x, n_bins=100):
    n = len(x)
    n_per_bin = int(n / n_bins)
    x_binned = np.zeros(n_bins)
    for i in range(n_bins):
        x_binned[i] = np.sum(x[i*n_per_bin:(i+1)*n_per_bin])
    return x_binned

In [ ]:
prior = utils.BoxUniform(
    low=torch.tensor([prior_cfg['logD_lims'][0], prior_cfg['dG_lims'][0]]),
    high=torch.tensor([prior_cfg['logD_lims'][1], prior_cfg['dG_lims'][1]]),
    device="cuda"
)

In [ ]:
fret_simulator = partial(
    sim.fret_simulator_simple,
    R0=4.8,
    kD=100,
    T=sim_cfg['total_time'],
    deltax=sim_cfg['deltax'],
    dt=sim_cfg['time_step'],
    saving_freq=sim_cfg['saving_frequency']
)

fret_simulator = process_simulator(fret_simulator, is_numpy_simulator=True, prior=prior)

In [ ]:
true_params = torch.tensor([[0.0, 4] for _ in range(1)])

In [ ]:
observation = fret_simulator(true_params)

In [ ]:
idx = 0
plt.figure(figsize=(7, 2.5))
plt.plot(bin_along_time(observation[idx, :,0].numpy(), n_bins=155), label="Donor")
plt.plot(bin_along_time(observation[idx, :,1].numpy(), n_bins=155), label="acceptor")
plt.ylabel("Photon count")
plt.xlabel("Time")


In [ ]:
with open("/home/dingeldein/Desktop/fret_sbi/fret_sbi_double_well_benchmarkt/outputs/2026-02-20/11-48-33/last_density_estimator.pkl", "rb") as f:
    density_estimator = pickle.load(f)

In [ ]:
posterior = DirectPosterior(density_estimator, prior=prior)

In [ ]:
samples = posterior.sample_batched((100000,), x=observation.cuda()).cpu().numpy()

In [ ]:
_ = analysis.pairplot(
    samples[:, 0, :],
    limits=[prior_cfg['logD_lims'], prior_cfg['dG_lims']],
    figsize=(7,7),
    points=true_params, 
    points_offdiag={'markersize': 6},
    points_colors='r', 
    labels=['$log(D)$', '$\Delta G$']
)

In [ ]:
for i in range(len(true_params)):
    plt.hist(samples[:, i, 1], bins=100, density=True, histtype='step', color='gray')
plt.vlines(true_params[0, 1], *plt.ylim(), color='r', label='True value')
plt.xlim(prior_cfg['dG_lims'])

In [ ]:
sim.double_well_integrator(T=1)
import time

In [ ]:
for i in range(3):
    t1 = time.time()
    sim.double_well_integrator(T=10)
    t2 = time.time()
    print(t2 - t1)